In [1]:
import json
from pathlib import Path

INPUT_DIR = Path("dataset")
OUTPUT_FILE = Path("dataset/dataset.jsonl")

count = 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:
    for file in sorted(INPUT_DIR.glob("*.json")):
        with open(file, "r", encoding="utf-8") as infile:
            obj = json.load(infile)
            outfile.write(json.dumps(obj))
            outfile.write("\n")
            count += 1

print(f"Saved {count} examples to {OUTPUT_FILE}")

Saved 29 examples to dataset.jsonl


In [9]:
from openai import OpenAI
from pathlib import Path
import datetime
import dotenv
import json
import os
import random
import re


def generate_dataset(
    N: int,
    filename: str = f"datasets/dataset-{datetime.date.today()}.jsonl",
):

    dotenv.load_dotenv()

    client = OpenAI(
        api_key=os.getenv("OPENAI_API_KEY")
    )

    TOOLS = [
        "classify_sequence",
        "search_pubmed",
        "gene_lookup",
        "final_answer",
    ]

    SYSTEM_PROMPT = """
You are generating training data for supervised fine-tuning.

Return ONLY valid JSON.

Schema:

{
  "messages": [
    {
      "role": "user",
      "content": "..."
    },
    {
      "role": "assistant",

      EITHER

      "content": "..."

      OR

      "tool_calls": [
        {
          "tool": "...",
          "arguments": {...}
        }
      ]
    }
  ]
}

Rules:

- Choose ONLY the requested tool.
- Use realistic biology questions.
- Questions should be diverse.
- For classify_sequence, include amino acid or nucleotide sequences.
- For search_pubmed, include realistic literature search queries.
- For gene_lookup, include realistic gene names.
- For final_answer, answer directly without tool_calls.
- Return JSON only.
"""

    output_file = Path(filename)
    output_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Resume automatically
    if output_file.exists():
        with open(output_file, "r", encoding="utf-8") as f:
            start = sum(1 for line in f if line.strip())
    else:
        start = 0

    print(f"Found {start} existing samples.")

    if start >= N:
        print("Dataset already complete.")
        return

    with open(output_file, "a", encoding="utf-8") as f:

        for i in range(start, N):

            tool = random.choice(TOOLS)

            prompt = f"""
Generate ONE training example.

Selected tool:

{tool}
"""

            response = client.responses.create(
                model="gpt-4o",
                input=[
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    },
                ],
            )

            text = response.output_text.strip()

            match = re.search(
                r"\{.*\}",
                text,
                flags=re.DOTALL,
            )

            if match:
                text = match.group(0)

            try:

                obj = json.loads(text)

                f.write(
                    json.dumps(
                        obj,
                        ensure_ascii=False,
                    )
                    + "\n"
                )

                f.flush()      # write immediately
                os.fsync(f.fileno())  # survive crashes/power loss

                print(f"{i+1}/{N} ✓ {tool}")

            except json.JSONDecodeError as e:

                print(f"\n{i+1}/{N} ✗ Invalid JSON")
                print(e)
                print("=" * 80)
                print(text)
                print("=" * 80)

In [1]:
#generate_dataset(500, "../datasets/v0.0.2/train.jsonl")

In [ ]:
baseline = {}

for p in prompts:

    ans = ask(p)

    baseline[p] = ans

    print("="*80)
    #print(p)
    print(ans)